# Capítulo 5: IA para la Automatización de la Respuesta a Incidentes *(Versión Simulada)*

## 5.1. Introducción

Esta versión genera **2,000 incidentes simulados** con niveles de severidad realistas. El clasificador SVM aprende a distinguir entre incidentes bajos, medios, altos y críticos basándose en características como hosts afectados, bytes exfiltrados y privilegios elevados.

---
## 5.2. Generación de datos simulados

In [ ]:
import numpy as np
import pandas as pd
import os

np.random.seed(42)
os.makedirs('data', exist_ok=True)

n = 2000
severities = np.random.choice([0,1,2,3], n, p=[0.40, 0.30, 0.20, 0.10])
records = []
for sev in severities:
    if sev == 0:
        row = {'num_hosts_afectados': np.random.randint(1,2), 'tipo_evento_cod': np.random.choice([1,2]),
               'bytes_exfiltrados': np.random.randint(0,500), 'duracion_seg': np.random.randint(1,30),
               'privilegios_elevados': 0}
    elif sev == 1:
        row = {'num_hosts_afectados': np.random.randint(1,5), 'tipo_evento_cod': np.random.choice([2,3,4]),
               'bytes_exfiltrados': np.random.randint(500,10000), 'duracion_seg': np.random.randint(15,120),
               'privilegios_elevados': np.random.choice([0,1], p=[0.7,0.3])}
    elif sev == 2:
        row = {'num_hosts_afectados': np.random.randint(3,15), 'tipo_evento_cod': np.random.choice([4,5,6]),
               'bytes_exfiltrados': np.random.randint(10000,500000), 'duracion_seg': np.random.randint(60,600),
               'privilegios_elevados': np.random.choice([0,1], p=[0.3,0.7])}
    else:
        row = {'num_hosts_afectados': np.random.randint(10,50), 'tipo_evento_cod': np.random.choice([5,6,7]),
               'bytes_exfiltrados': np.random.randint(500000,5000000), 'duracion_seg': np.random.randint(300,3600),
               'privilegios_elevados': 1}
    row['severity'] = sev
    records.append(row)

df_inc = pd.DataFrame(records).sample(frac=1, random_state=42).reset_index(drop=True)
df_inc.to_csv('data/incident_data.csv', index=False)
print(f"[OK] data/incident_data.csv: {len(df_inc)} registros")
print(f"     Distribución: {df_inc['severity'].value_counts().sort_index().to_dict()}")
df_inc.head()

---
## 5.3. Triaje automático con SVM

In [ ]:
# Listing 5.1: Clasificación de incidentes por severidad con SVM

from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

incidents = pd.read_csv('data/incident_data.csv').dropna()
X = incidents.drop('severity', axis=1)
y = incidents['severity']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42))
])
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
etiquetas = ['Bajo', 'Medio', 'Alto', 'Crítico']
print("=== Reporte de clasificación ===")
print(classification_report(y_test, y_pred, target_names=etiquetas))

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=etiquetas).plot(cmap='Reds')
plt.title('Triaje SVM (Simulado)')
plt.tight_layout()
plt.savefig('data/confusion_matrix_svm_incidents.png', dpi=150)
plt.show()

In [ ]:
# Predicción de nuevo incidente
nuevo = pd.DataFrame([{'num_hosts_afectados':3,'tipo_evento_cod':5,
                       'bytes_exfiltrados':102400,'duracion_seg':120,'privilegios_elevados':1}])
proba = pipeline.predict_proba(nuevo)[0]
pred  = pipeline.predict(nuevo)[0]
print(f"Severidad predicha: {etiquetas[int(pred)]}")
for i, p in enumerate(proba):
    print(f"  P({etiquetas[i]}) = {p:.3f}")

## 5.4. Motor de orquestación (modo simulado)

In [ ]:
# Listing 5.2 (simplificado - modo simulado activo)
import logging
from dataclasses import dataclass
from enum import IntEnum

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)-8s %(message)s')

class Severidad(IntEnum):
    BAJO=0; MEDIO=1; ALTO=2; CRITICO=3

@dataclass
class Incidente:
    id_sistema: str; ip_origen: str; severidad: Severidad; descripcion: str

def orquestar_respuesta(inc: Incidente):
    logging.info(f"[Incidente] {inc.id_sistema} | {inc.severidad.name}")
    acciones = {Severidad.BAJO: ['Registrar y monitorear'],
                Severidad.MEDIO: ['Notificar equipo [SIMULADO]'],
                Severidad.ALTO:  ['Bloquear IP [SIMULADO]', 'Notificar equipo [SIMULADO]'],
                Severidad.CRITICO: ['Aislar sistema [SIMULADO]','Bloquear IP [SIMULADO]','Notificar [SIMULADO]']}
    for accion in acciones[inc.severidad]:
        logging.info(f"  → {accion}")

for nombre, sev in [('WS-USER-101', Severidad.BAJO), ('WS-DEV-205', Severidad.MEDIO),
                    ('DB-PROD-003', Severidad.ALTO), ('SRV-PROD-042', Severidad.CRITICO)]:
    orquestar_respuesta(Incidente(nombre, '10.0.0.1', sev, 'Incidente simulado'))
    print('---')

## 5.5. Guardado de artefactos

In [ ]:
import joblib
os.makedirs('models', exist_ok=True)
joblib.dump(pipeline, 'models/svm_incident_triage.pkl')
print("[OK] models/svm_incident_triage.pkl")